Audyt metodologii 2026-09-10: wyniki historyczne unieważnione. Definicje i ograniczenia: `../docs/methodology_audit.md`. Przeliczenia lokalne: `../data/processed/audit_v2/`.


In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

MODEL_CLEAN_PATH = DATA_PROCESSED / "df_model_clean_v1.parquet"
TAG_CLASSIFICATION_PATH = DATA_PROCESSED / "tag_classification_v1.xlsx"

df = pd.read_parquet(MODEL_CLEAN_PATH)
classified = pd.read_excel(TAG_CLASSIFICATION_PATH)

print("df:", df.shape)
print("classified:", classified.shape)
print(df.index.min(), "->", df.index.max())

df.head()
import sys
sys.path.insert(0, str(PROJECT_ROOT))
from src.time_analysis import (validate_time, time_shift, past_mean, lagged_corr,
    rapping_starts, rapping_features, complete_window, tail_mask as post_tail_mask, coverage)

validate_time(df)
AUDIT_OUTPUT = DATA_PROCESSED / "audit_v2"
AUDIT_OUTPUT.mkdir(exist_ok=True)


In [ ]:
target_col = "008A01345"  # Pył BC1 Stężenie aktualne

rapping_tags = classified.loc[
    classified["category"] == "esp_rapping",
    "tag"
].tolist()

rapping_info = classified[classified["tag"].isin(rapping_tags)][
    ["tag", "description", "symbol", "unit", "category"]
].copy()

print("Liczba tagów strzepywaczy:", len(rapping_tags))
rapping_info

In [ ]:
rapping_value_summary = []

for col in rapping_tags:
    rapping_value_summary.append({
        "tag": col,
        "description": classified.loc[classified["tag"] == col, "description"].iloc[0],
        "n_unique": df[col].nunique(dropna=True),
        "unique_values": sorted(df[col].dropna().unique())[:20],
        "min": df[col].min(),
        "max": df[col].max(),
        "mean": df[col].mean()
    })

rapping_value_summary = pd.DataFrame(rapping_value_summary)

rapping_value_summary

In [ ]:
# Wybieramy tylko realne sygnały aktywacji 0/1
rapping_event_tags = rapping_value_summary.loc[
    rapping_value_summary["n_unique"] > 1,
    "tag"
].tolist()

print("Rapping event tags:", rapping_event_tags)

df_rap = df.copy()

# jakikolwiek strzepywacz aktywny
df_rap["rapping_any"] = (df_rap[rapping_event_tags].sum(axis=1) > 0).astype(int)

# strzepywanie elektrod zbiorczych
collecting_tags = classified[
    classified["description"].str.contains("zbior", case=False, na=False)
    & classified["tag"].isin(rapping_event_tags)
]["tag"].tolist()

# strzepywanie elektrod ulotowych
discharge_tags = classified[
    classified["description"].str.contains("Ulotowych|ULOT", case=False, na=False)
    & classified["tag"].isin(rapping_event_tags)
]["tag"].tolist()

df_rap["rapping_collecting"] = (df_rap[collecting_tags].sum(axis=1) > 0).astype(int)
df_rap["rapping_discharge"] = (df_rap[discharge_tags].sum(axis=1) > 0).astype(int)

print("Collecting electrode rapping tags:", collecting_tags)
print("Discharge electrode rapping tags:", discharge_tags)

df_rap[["rapping_any", "rapping_collecting", "rapping_discharge", target_col]].head()

In [ ]:
rapping_summary = pd.DataFrame({
    "signal": ["rapping_any", "rapping_collecting", "rapping_discharge"],
    "active_samples": [
        df_rap["rapping_any"].sum(),
        df_rap["rapping_collecting"].sum(),
        df_rap["rapping_discharge"].sum()
    ],
    "active_percent": [
        df_rap["rapping_any"].mean() * 100,
        df_rap["rapping_collecting"].mean() * 100,
        df_rap["rapping_discharge"].mean() * 100
    ]
})

rapping_summary

In [ ]:
rapping_dust_summary = (
    df_rap
    .groupby("rapping_any")[target_col]
    .agg(
        n_samples="count",
        mean="mean",
        median="median",
        std="std",
        min="min",
        max="max",
        q90=lambda x: x.quantile(0.90),
        q95=lambda x: x.quantile(0.95),
        q99=lambda x: x.quantile(0.99),
    )
    .reset_index()
)

rapping_dust_summary

In [ ]:
for signal in ["rapping_collecting", "rapping_discharge"]:
    print("\n", signal)

    summary = (
        df_rap
        .groupby(signal)[target_col]
        .agg(
            n_samples="count",
            mean="mean",
            median="median",
            std="std",
            min="min",
            max="max",
            q90=lambda x: x.quantile(0.90),
            q95=lambda x: x.quantile(0.95),
            q99=lambda x: x.quantile(0.99),
        )
        .reset_index()
    )

    display(summary)

In [ ]:
thresholds = [10, 20, 40]

rows = []

for signal in ["rapping_any", "rapping_collecting", "rapping_discharge"]:
    for state in [0, 1]:
        subset = df_rap[df_rap[signal] == state]

        row = {
            "signal": signal,
            "state": state,
            "n_samples": len(subset),
            "mean_dust": subset[target_col].mean(),
        }

        for threshold in thresholds:
            row[f"P_dust_gt_{threshold}"] = (subset[target_col] > threshold).mean() * 100

        rows.append(row)

rapping_peak_prob = pd.DataFrame(rows)

rapping_peak_prob

In [ ]:
# Wykrywanie zdarzeń startu strzepywania: przejście 0 -> 1

event_rows = []

for tag in rapping_event_tags:
    signal = df_rap[tag]
    starts = rapping_starts(signal)

    tag_info = classified.loc[classified["tag"] == tag].iloc[0]

    event_rows.append({
        "tag": tag,
        "description": tag_info["description"],
        "n_start_events": starts.sum(),
        "active_percent": signal.mean() * 100
    })

rapping_events_summary = pd.DataFrame(event_rows).sort_values("n_start_events", ascending=False)

rapping_events_summary

In [ ]:
# Analiza maksymalnego pyłu po starcie strzepywania

window_defs = {
    "0_1_min": (0, 6),      # 0-60 s
    "1_3_min": (6, 18),     # 60-180 s
    "3_5_min": (18, 30),    # 180-300 s
    "5_10_min": (30, 60),   # 300-600 s
}

event_analysis_rows = []

for tag in rapping_event_tags:
    signal = df_rap[tag]
    starts = (rapping_starts(signal))
    event_times = df_rap.index[starts]

    description = classified.loc[classified["tag"] == tag, "description"].iloc[0]

    for event_time in event_times:
        event_pos = df_rap.index.get_loc(event_time)

        row = {
            "event_time": event_time,
            "tag": tag,
            "description": description,
            "dust_at_event": df_rap.iloc[event_pos][target_col],
        }

        for window_name, (start_offset, end_offset) in window_defs.items():
            # Require continuity from event through the requested window boundary.
            full = complete_window(df_rap[[target_col]], event_time, 0, end_offset * 10)
            window_dust = (full.iloc[start_offset:end_offset][target_col]
                           if full is not None else pd.Series(dtype=float))
            row[f"max_dust_{window_name}"] = window_dust.max()
            row[f"mean_dust_{window_name}"] = window_dust.mean()

        event_analysis_rows.append(row)

rapping_event_analysis = pd.DataFrame(event_analysis_rows)

print(rapping_event_analysis.shape)
rapping_event_analysis.head()

In [ ]:
summary_cols = [
    "dust_at_event",
    "max_dust_0_1_min",
    "max_dust_1_3_min",
    "max_dust_3_5_min",
    "max_dust_5_10_min",
]

rapping_window_summary = (
    rapping_event_analysis
    .groupby(["tag", "description"])[summary_cols]
    .agg(["count", "mean", "median", "max"])
)

rapping_window_summary

In [ ]:
thresholds = [20, 40, 60]

prob_rows = []

for tag in rapping_event_tags:
    tag_events = rapping_event_analysis[rapping_event_analysis["tag"] == tag]
    description = classified.loc[classified["tag"] == tag, "description"].iloc[0]

    for window_name in ["0_1_min", "1_3_min", "3_5_min", "5_10_min"]:
        max_col = f"max_dust_{window_name}"

        row = {
            "tag": tag,
            "description": description,
            "window": window_name,
            "n_events": tag_events[max_col].notna().sum(),
            "mean_max_dust": tag_events[max_col].mean(),
            "median_max_dust": tag_events[max_col].median(),
        }

        for threshold in thresholds:
            row[f"P_max_dust_gt_{threshold}"] = (
                tag_events[max_col].dropna() > threshold
            ).mean() * 100

        prob_rows.append(row)

rapping_window_prob = pd.DataFrame(prob_rows)

rapping_window_prob.sort_values(
    ["P_max_dust_gt_40", "P_max_dust_gt_20"],
    ascending=False
)

In [ ]:
event_tag = "008B05154"

signal = df_rap[event_tag]
starts = rapping_starts(signal)
event_times = df_rap.index[starts]

pre_steps = 12    # 2 min przed, bo 12 * 10 s
post_steps = 30   # 5 min po, bo 30 * 10 s

profiles = []

for event_time in event_times:
    window = complete_window(df_rap[[target_col]], event_time, -pre_steps * 10, post_steps * 10)
    if window is not None:
        profiles.append(window[target_col].to_numpy())

if not profiles:
    raise ValueError("No complete rapping profiles")
profiles = np.array(profiles)

time_axis_min = np.arange(-pre_steps, post_steps + 1) * 10 / 60

mean_profile = profiles.mean(axis=0)
median_profile = np.median(profiles, axis=0)
q25_profile = np.quantile(profiles, 0.25, axis=0)
q75_profile = np.quantile(profiles, 0.75, axis=0)

print("Number of events used:", profiles.shape[0])

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(time_axis_min, mean_profile, label="Mean dust")
plt.plot(time_axis_min, median_profile, label="Median dust")
plt.fill_between(time_axis_min, q25_profile, q75_profile, alpha=0.3, label="25–75 percentile")

plt.axvline(0, linestyle="--", label="Rapping start")

plt.title("Dust concentration around rapping start - collecting electrode zone 3")
plt.xlabel("Time relative to rapping start [min]")
plt.ylabel("Dust concentration [mg/Nm3]")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
def compute_event_profile(df, event_tag, target_col, pre_steps=12, post_steps=30):
    signal = df[event_tag]
    starts = rapping_starts(signal)
    event_times = df.index[starts]

    profiles = []

    for event_time in event_times:
        window = complete_window(df[[target_col]], event_time, -pre_steps * 10, post_steps * 10)
        if window is not None:
            profiles.append(window[target_col].to_numpy())

    if len(profiles) == 0:
        return None

    profiles = np.array(profiles)

    return {
        "tag": event_tag,
        "n_events": profiles.shape[0],
        "mean_profile": profiles.mean(axis=0),
        "median_profile": np.median(profiles, axis=0),
        "q25_profile": np.quantile(profiles, 0.25, axis=0),
        "q75_profile": np.quantile(profiles, 0.75, axis=0),
    }

In [ ]:
pre_steps = 12
post_steps = 30
time_axis_min = np.arange(-pre_steps, post_steps + 1) * 10 / 60

plt.figure(figsize=(13, 6))

for tag in rapping_event_tags:
    result = compute_event_profile(
        df=df_rap,
        event_tag=tag,
        target_col=target_col,
        pre_steps=pre_steps,
        post_steps=post_steps
    )

    if result is None:
        continue

    description = classified.loc[classified["tag"] == tag, "description"].iloc[0]

    plt.plot(
        time_axis_min,
        result["median_profile"],
        label=f"{tag} - {description}"
    )

plt.axvline(0, linestyle="--", label="Rapping start")

plt.title("Median dust concentration around rapping start - all rapping signals")
plt.xlabel("Time relative to rapping start [min]")
plt.ylabel("Dust concentration [mg/Nm3]")
plt.legend(fontsize=8)
plt.grid(True)
plt.show()

In [ ]:
profile_summary_rows = []

pre_steps = 12
post_steps = 30

for tag in rapping_event_tags:
    result = compute_event_profile(
        df=df_rap,
        event_tag=tag,
        target_col=target_col,
        pre_steps=pre_steps,
        post_steps=post_steps
    )

    if result is None:
        continue

    description = classified.loc[classified["tag"] == tag, "description"].iloc[0]

    median_profile = result["median_profile"]
    mean_profile = result["mean_profile"]

    # baseline: mediana/średnia z okresu 2 min przed startem
    baseline_median = np.median(median_profile[:pre_steps])
    baseline_mean = np.mean(mean_profile[:pre_steps])

    # peak po starcie, tylko część od t=0 do +5 min
    post_median_profile = median_profile[pre_steps:]
    post_mean_profile = mean_profile[pre_steps:]

    peak_median = np.max(post_median_profile)
    peak_mean = np.max(post_mean_profile)

    peak_median_idx = np.argmax(post_median_profile)
    peak_mean_idx = np.argmax(post_mean_profile)

    time_to_peak_median_min = peak_median_idx * 10 / 60
    time_to_peak_mean_min = peak_mean_idx * 10 / 60

    profile_summary_rows.append({
        "tag": tag,
        "description": description,
        "n_events": result["n_events"],

        "baseline_median": baseline_median,
        "peak_median": peak_median,
        "delta_median": peak_median - baseline_median,
        "time_to_peak_median_min": time_to_peak_median_min,

        "baseline_mean": baseline_mean,
        "peak_mean": peak_mean,
        "delta_mean": peak_mean - baseline_mean,
        "time_to_peak_mean_min": time_to_peak_mean_min,
    })

profile_summary = (
    pd.DataFrame(profile_summary_rows)
    .sort_values("delta_median", ascending=False)
    .reset_index(drop=True)
)

profile_summary

In [ ]:
RAPPING_EVENT_ANALYSIS_PATH = AUDIT_OUTPUT / "rapping_event_analysis_v1.parquet"
RAPPING_WINDOW_PROB_PATH = AUDIT_OUTPUT / "rapping_window_probability_v1.csv"
RAPPING_PROFILE_SUMMARY_PATH = AUDIT_OUTPUT / "rapping_profile_summary_v1.csv"

rapping_event_analysis.to_parquet(RAPPING_EVENT_ANALYSIS_PATH)
rapping_window_prob.to_csv(RAPPING_WINDOW_PROB_PATH, index=False)
profile_summary.to_csv(RAPPING_PROFILE_SUMMARY_PATH, index=False)

print(RAPPING_EVENT_ANALYSIS_PATH)
print(RAPPING_WINDOW_PROB_PATH)
print(RAPPING_PROFILE_SUMMARY_PATH)

In [ ]:
critical_rapping_tag = "008B05154"

df_rap_time = df_rap.copy()

signal = df_rap_time[critical_rapping_tag]

# start zdarzenia = przejście 0 -> 1
df_rap_time["rapping_3_collecting_start"] = (
    rapping_starts(signal)
).astype(int)

print("Number of starts:", df_rap_time["rapping_3_collecting_start"].sum())

In [ ]:
# Timestamp-derived features; unknown immediately after gaps remains NaN.
corrected_rapping = rapping_features(signal)
df_rap_time[corrected_rapping.columns] = corrected_rapping


In [ ]:
df_rap_time[corrected_rapping.columns].describe()

In [ ]:
time_bins = np.arange(0, 5.5, 0.5)

df_rap_time["time_since_rapping_bin"] = pd.cut(
    df_rap_time["minutes_since_rapping_3_collecting_start_0_5"],
    bins=time_bins,
    include_lowest=True
)

dust_by_time_since_rapping = (
    df_rap_time
    .dropna(subset=["time_since_rapping_bin"])
    .groupby("time_since_rapping_bin", observed=True)[target_col]
    .agg(
        n_samples="count",
        mean="mean",
        median="median",
        q25=lambda x: x.quantile(0.25),
        q75=lambda x: x.quantile(0.75),
        max="max"
    )
    .reset_index()
)

dust_by_time_since_rapping

In [ ]:
plt.figure(figsize=(10, 5))

x = dust_by_time_since_rapping["time_since_rapping_bin"].astype(str)

plt.plot(x, dust_by_time_since_rapping["mean"], marker="o", label="Mean")
plt.plot(x, dust_by_time_since_rapping["median"], marker="o", label="Median")

plt.title("Dust concentration vs time since collecting electrode zone 3 rapping start")
plt.xlabel("Time since rapping start [min]")
plt.ylabel("Dust concentration [mg/Nm3]")
plt.xticks(rotation=45)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
rapping_feature_cols = corrected_rapping.columns.tolist()
df_rapping_features = corrected_rapping.copy()
df_rapping_features.head()


In [ ]:
df_rapping_features.describe()

In [ ]:
df_rapping_features.isna().sum()

In [ ]:
X_PATH = DATA_PROCESSED / "X_feature_engineered_v1.parquet"
Y_PATH = DATA_PROCESSED / "y_feature_engineered_v1.parquet"

X = pd.read_parquet(X_PATH)
y = pd.read_parquet(Y_PATH)[target_col]

print("X:", X.shape)
print("y:", y.shape)
print("X index:", X.index.min(), "->", X.index.max())
print("Rapping features index:", df_rapping_features.index.min(), "->", df_rapping_features.index.max())

In [ ]:
X_rapping = X.copy()
X_rapping = X_rapping.join(df_rapping_features, how="left")

In [ ]:
print("X_rapping:", X_rapping.shape)
print(X_rapping[rapping_feature_cols].isna().sum())

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score

split_idx = int(len(X_rapping) * 0.8)

X_train_rapping = X_rapping.iloc[:split_idx]
X_test_rapping = X_rapping.iloc[split_idx:]

y_train_rapping = y.iloc[:split_idx]
y_test_rapping = y.iloc[split_idx:]

model_rapping = XGBRegressor(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

model_rapping.fit(X_train_rapping, y_train_rapping)

y_pred_rapping = model_rapping.predict(X_test_rapping)

mae_rapping = mean_absolute_error(y_test_rapping, y_pred_rapping)
r2_rapping = r2_score(y_test_rapping, y_pred_rapping)

print(f"MAE rapping features: {mae_rapping:.3f}")
print(f"R2 rapping features: {r2_rapping:.3f}")

In [ ]:
eval_rapping_df = pd.DataFrame({
    "y_true": y_test_rapping,
    "y_pred": y_pred_rapping
}, index=y_test_rapping.index)

eval_rapping_df["error"] = eval_rapping_df["y_true"] - eval_rapping_df["y_pred"]
eval_rapping_df["abs_error"] = eval_rapping_df["error"].abs()

bins = [0, 10, 20, 40, np.inf]
labels = ["0-10", "10-20", "20-40", ">40"]

eval_rapping_df["dust_range"] = pd.cut(
    eval_rapping_df["y_true"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

range_metrics_rapping = (
    eval_rapping_df
    .groupby("dust_range", observed=True)
    .apply(lambda g: pd.Series({
        "n_samples": len(g),
        "mean_y_true": g["y_true"].mean(),
        "mean_y_pred": g["y_pred"].mean(),
        "MAE": mean_absolute_error(g["y_true"], g["y_pred"]),
        "RMSE": np.sqrt(np.mean((g["y_true"] - g["y_pred"]) ** 2)),
        "bias": (g["y_pred"] - g["y_true"]).mean(),
        "max_abs_error": g["abs_error"].max()
    }))
    .reset_index()
)

range_metrics_rapping

In [ ]:
# Model referencyjny: pełny feature engineering bez nowych cech strzepywacza

split_idx = int(len(X) * 0.8)

X_train_full = X.iloc[:split_idx]
X_test_full = X.iloc[split_idx:]

y_train_full = y.iloc[:split_idx]
y_test_full = y.iloc[split_idx:]

model_full_ref = XGBRegressor(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

model_full_ref.fit(X_train_full, y_train_full)

y_pred_full_ref = model_full_ref.predict(X_test_full)

print("Reference full FE:")
print("MAE:", mean_absolute_error(y_test_full, y_pred_full_ref))
print("R2:", r2_score(y_test_full, y_pred_full_ref))

In [ ]:
test_index = y_test_rapping.index

critical_window_test = df_rapping_features.loc[test_index, [
    "minutes_since_rapping_3_collecting_start_0_5",
    "is_within_5min_after_rapping_3_collecting"
]].copy()

eval_compare = pd.DataFrame({
    "y_true": y_test_rapping,
    "y_pred_full_fe": y_pred_full_ref,
    "y_pred_rapping": y_pred_rapping,
}, index=y_test_rapping.index)

eval_compare = eval_compare.join(critical_window_test)

eval_compare["abs_error_full_fe"] = (
    eval_compare["y_true"] - eval_compare["y_pred_full_fe"]
).abs()

eval_compare["abs_error_rapping"] = (
    eval_compare["y_true"] - eval_compare["y_pred_rapping"]
).abs()

# okno 0–3 min po starcie krytycznego strzepywacza
mask_0_3min = (
    (eval_compare["minutes_since_rapping_3_collecting_start_0_5"] >= 0) &
    (eval_compare["minutes_since_rapping_3_collecting_start_0_5"] <= 3)
)

compare_0_3min = eval_compare[mask_0_3min]

summary_0_3min = pd.Series({
    "n_samples": len(compare_0_3min),
    "mean_y_true": compare_0_3min["y_true"].mean(),
    "mean_pred_full_fe": compare_0_3min["y_pred_full_fe"].mean(),
    "mean_pred_rapping": compare_0_3min["y_pred_rapping"].mean(),
    "MAE_full_fe": compare_0_3min["abs_error_full_fe"].mean(),
    "MAE_rapping": compare_0_3min["abs_error_rapping"].mean(),
    "bias_full_fe": (compare_0_3min["y_pred_full_fe"] - compare_0_3min["y_true"]).mean(),
    "bias_rapping": (compare_0_3min["y_pred_rapping"] - compare_0_3min["y_true"]).mean(),
    "P_y_true_gt_20": (compare_0_3min["y_true"] > 20).mean() * 100,
    "P_y_true_gt_40": (compare_0_3min["y_true"] > 40).mean() * 100,
})

summary_0_3min

In [ ]:
# Okno zakłócone: 0–3 min po starcie strzepywania elektrod zbiorczych strefy 3

critical_time_col = "minutes_since_rapping_3_collecting_start_0_5"

rapping_time_for_X = df_rapping_features.loc[X.index, critical_time_col]

mask_after_rapping_0_3min = (
    (rapping_time_for_X >= 0) &
    (rapping_time_for_X <= 3)
)

print("Samples total:", len(X))
print("Samples in 0-3 min after critical rapping:", mask_after_rapping_0_3min.sum())
print("Percent removed:", round(mask_after_rapping_0_3min.mean() * 100, 3), "%")

In [ ]:
X_no_rapping_peak = X.loc[(~mask_after_rapping_0_3min) & rapping_time_for_X.notna()].copy()
y_no_rapping_peak = y.loc[(~mask_after_rapping_0_3min) & rapping_time_for_X.notna()].copy()

print("X without rapping peak periods:", X_no_rapping_peak.shape)
print("y without rapping peak periods:", y_no_rapping_peak.shape)

print("Original y mean:", y.mean())
print("Filtered y mean:", y_no_rapping_peak.mean())

print("Original y max:", y.max())
print("Filtered y max:", y_no_rapping_peak.max())

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np

split_idx = int(len(X_no_rapping_peak) * 0.8)

X_train_no_peak = X_no_rapping_peak.iloc[:split_idx]
X_test_no_peak = X_no_rapping_peak.iloc[split_idx:]

y_train_no_peak = y_no_rapping_peak.iloc[:split_idx]
y_test_no_peak = y_no_rapping_peak.iloc[split_idx:]

model_no_peak = XGBRegressor(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

model_no_peak.fit(X_train_no_peak, y_train_no_peak)

y_pred_no_peak = model_no_peak.predict(X_test_no_peak)

mae_no_peak = mean_absolute_error(y_test_no_peak, y_pred_no_peak)
r2_no_peak = r2_score(y_test_no_peak, y_pred_no_peak)

print(f"MAE without 0-3 min rapping peak periods: {mae_no_peak:.3f}")
print(f"R2 without 0-3 min rapping peak periods: {r2_no_peak:.3f}")

In [ ]:
eval_no_peak_df = pd.DataFrame({
    "y_true": y_test_no_peak,
    "y_pred": y_pred_no_peak
}, index=y_test_no_peak.index)

eval_no_peak_df["error"] = eval_no_peak_df["y_true"] - eval_no_peak_df["y_pred"]
eval_no_peak_df["abs_error"] = eval_no_peak_df["error"].abs()

bins = [0, 10, 20, 40, np.inf]
labels = ["0-10", "10-20", "20-40", ">40"]

eval_no_peak_df["dust_range"] = pd.cut(
    eval_no_peak_df["y_true"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

range_metrics_no_peak = (
    eval_no_peak_df
    .groupby("dust_range", observed=True)
    .apply(lambda g: pd.Series({
        "n_samples": len(g),
        "mean_y_true": g["y_true"].mean(),
        "mean_y_pred": g["y_pred"].mean(),
        "MAE": mean_absolute_error(g["y_true"], g["y_pred"]),
        "RMSE": np.sqrt(np.mean((g["y_true"] - g["y_pred"]) ** 2)),
        "bias": (g["y_pred"] - g["y_true"]).mean(),
        "max_abs_error": g["abs_error"].max()
    }))
    .reset_index()
)

range_metrics_no_peak

In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(
    y_test_no_peak.index,
    y_test_no_peak.values,
    label="Actual",
    linewidth=1
)

plt.plot(
    y_test_no_peak.index,
    y_pred_no_peak,
    label="Predicted - no rapping peak periods",
    linewidth=1
)

plt.title("Dust concentration prediction after removing 0–3 min periods after 008B05154 rapping")
plt.xlabel("Time")
plt.ylabel("Dust concentration [mg/Nm3]")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
residuals_no_peak = y_test_no_peak - y_pred_no_peak

plt.figure(figsize=(14, 4))

plt.plot(
    y_test_no_peak.index,
    residuals_no_peak,
    linewidth=1
)

plt.axhline(0, linestyle="--")

plt.title("Prediction residuals after removing 0–3 min periods after 008B05154 rapping")
plt.xlabel("Time")
plt.ylabel("Residual: actual - predicted [mg/Nm3]")
plt.grid(True)
plt.show()

In [ ]:
# Compare on the SAME timestamps; do not copy historical metrics.
common = eval_no_peak_df.index.intersection(eval_compare.index)
ref = eval_compare.loc[common].copy()
ref['dust_range'] = pd.cut(ref.y_true, bins=bins, labels=labels, include_lowest=True)
range_metrics_full = ref.groupby('dust_range', observed=True).agg(
    MAE_full_FE=('abs_error_full_fe', 'mean'), n_samples_full_FE=('y_true', 'size'))
range_metrics_full['bias_full_FE'] = (ref.y_pred_full_fe - ref.y_true).groupby(ref.dust_range, observed=True).mean()
range_metrics_compare = range_metrics_full.reset_index().merge(
    range_metrics_no_peak[['dust_range', 'n_samples', 'MAE', 'bias']], on='dust_range', how='left'
).rename(columns={'n_samples':'n_samples_no_peak', 'MAE':'MAE_no_peak', 'bias':'bias_no_peak'})
# Different training cutoffs remain a limitation; this is exploratory only.
range_metrics_compare


In [ ]:
x = np.arange(len(range_metrics_compare))
width = 0.35

plt.figure(figsize=(9, 5))

plt.bar(
    x - width/2,
    range_metrics_compare["MAE_full_FE"],
    width,
    label="Full FE"
)

plt.bar(
    x + width/2,
    range_metrics_compare["MAE_no_peak"],
    width,
    label="After removing rapping peak periods"
)

plt.xticks(x, range_metrics_compare["dust_range"])
plt.title("MAE by dust range before and after removing 008B05154 rapping periods")
plt.xlabel("Dust concentration range [mg/Nm3]")
plt.ylabel("MAE [mg/Nm3]")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))

plt.bar(
    x - width/2,
    range_metrics_compare["n_samples_full_FE"],
    width,
    label="Full FE"
)

plt.bar(
    x + width/2,
    range_metrics_compare["n_samples_no_peak"],
    width,
    label="After removing rapping peak periods"
)

plt.xticks(x, range_metrics_compare["dust_range"])
plt.title("Number of test samples by dust range before and after removing 008B05154 rapping periods")
plt.xlabel("Dust concentration range [mg/Nm3]")
plt.ylabel("Number of samples")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))

plt.bar(
    x - width/2,
    range_metrics_compare["bias_full_FE"],
    width,
    label="Full FE"
)

plt.bar(
    x + width/2,
    range_metrics_compare["bias_no_peak"],
    width,
    label="After removing rapping peak periods"
)

plt.axhline(0, linestyle="--")

plt.xticks(x, range_metrics_compare["dust_range"])
plt.title("Prediction bias by dust range before and after removing 008B05154 rapping periods")
plt.xlabel("Dust concentration range [mg/Nm3]")
plt.ylabel("Bias: mean(predicted - actual) [mg/Nm3]")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
RAPPING_FEATURES_PATH = AUDIT_OUTPUT / "rapping_features_v1.parquet"

df_rapping_features.to_parquet(RAPPING_FEATURES_PATH)

print(RAPPING_FEATURES_PATH)
print(df_rapping_features.shape)

In [ ]:
BASE_EMISSION_MASK_PATH = AUDIT_OUTPUT / "base_emission_mask_no_008B05154_0_3min_v1.parquet"

base_emission_mask = pd.DataFrame({
    "is_base_emission": (~mask_after_rapping_0_3min) & rapping_time_for_X.notna()
}, index=X.index)

base_emission_mask.to_parquet(BASE_EMISSION_MASK_PATH)

print(BASE_EMISSION_MASK_PATH)
print(base_emission_mask["is_base_emission"].value_counts())

In [ ]:
X_BASE_PATH = AUDIT_OUTPUT / "X_feature_engineered_base_emission_v1.parquet"
Y_BASE_PATH = AUDIT_OUTPUT / "y_feature_engineered_base_emission_v1.parquet"

X_no_rapping_peak.to_parquet(X_BASE_PATH)
y_no_rapping_peak.to_frame(name=target_col).to_parquet(Y_BASE_PATH)

print(X_BASE_PATH, X_no_rapping_peak.shape)
print(Y_BASE_PATH, y_no_rapping_peak.shape)

In [ ]:
NO_PEAK_RESULTS_PATH = AUDIT_OUTPUT / "model_results_base_emission_v1.csv"

model_results_base_emission = pd.DataFrame([
    {
        "model": "xgboost_full_fe_no_008B05154_0_3min",
        "description": "Full FE model after removing 0-3 min periods after collecting electrode zone 3 rapping",
        "MAE": mae_no_peak,
        "R2": r2_no_peak,
        "n_samples": len(X_no_rapping_peak)
    }
])

model_results_base_emission.to_csv(NO_PEAK_RESULTS_PATH, index=False)

print(NO_PEAK_RESULTS_PATH)
model_results_base_emission